In [ ]:
forget the report its previous

here is full code
!pip install pennylane pennylane-lightning-gpu --upgrade
!pip install custatevec-cu12 cuquantum-cu12

# =========================
# IMPORTS
# =========================
import torch
import torch.nn as nn
import pennylane as qml
from torch.utils.data import DataLoader, TensorDataset
from tqdm import tqdm
import math

# =========================
# CONFIG
# =========================
torch.manual_seed(42)

n_qubits = 6
n_layers = 2          # reduced
n_steps = 2           # evolution steps
input_dim = 10
hidden_dim = 128
vocab_size = 10
batch_size = 16       # reduced
epochs = 12

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# =========================
# DATASET
# =========================
def generate_data(n=6000, train=True):

    low, high = (0, 20) if train else (21, 40)

    a = torch.randint(low, high, (n,))
    b = torch.randint(low, high, (n,))
    task = torch.randint(0, 3, (n,))

    y = torch.zeros(n)

    for i in range(n):
        if task[i] == 0:
            y[i] = a[i] + b[i]
        elif task[i] == 1:
            y[i] = a[i] * b[i]
        else:
            y[i] = a[i] + b[i] if a[i] > b[i] else a[i] * b[i]

    y = (y % 10).long()

    x_raw = torch.stack([a, b], dim=1).float() / 40.0
    task_onehot = torch.nn.functional.one_hot(task, num_classes=3).float()

    x_raw = torch.cat([x_raw, task_onehot], dim=1)

    x = torch.cat([
        torch.sin(math.pi * x_raw),
        torch.cos(math.pi * x_raw)
    ], dim=1)

    return x, y

x_train, y_train = generate_data(6000, True)
x_test, y_test = generate_data(2000, False)

train_loader = DataLoader(TensorDataset(x_train, y_train), batch_size=batch_size, shuffle=True)
test_loader = DataLoader(TensorDataset(x_test, y_test), batch_size=batch_size)


# =========================
# QUANTUM DEVICE
# =========================
dev = qml.device("lightning.gpu", wires=n_qubits)

# =========================
# QNODE (REQUIRED)
# =========================
@qml.qnode(dev, interface="torch", diff_method="adjoint")
def qnode(inputs, weights):

    qml.AngleEmbedding(inputs, wires=range(n_qubits))
    qml.StronglyEntanglingLayers(weights, wires=range(n_qubits))

    return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]

# =========================
# CLASSICAL MODEL
# =========================
class ClassicalModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, vocab_size)
        )

    def forward(self, x):
        return self.net(x.to(device))

class ModelV81(nn.Module):
    def __init__(self):
        super().__init__()

        # ===== ENCODER =====
        self.encoder = nn.Linear(input_dim, n_qubits)

        # ===== MIX (EVOLUTION STEP) =====
        self.mix = nn.Sequential(
            nn.Linear(n_qubits * 2, n_qubits),
            nn.ReLU()
        )

        # ===== QUANTUM WEIGHTS =====
        self.q_weights = nn.Parameter(
            0.01 * torch.randn(n_steps, n_layers, n_qubits, 3)
        )

        # ===== QUANTUM DECODER =====
        self.decoder = nn.Sequential(
            nn.Linear(n_qubits, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, vocab_size)
        )

        # ===== CLASSICAL PATH =====
        self.classical_head = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, vocab_size)
        )

        # ===== ALPHA (LEARNABLE BLENDING) =====
        self.alpha = nn.Parameter(torch.tensor(-1.0))

    def forward(self, x):

        x = x.to(device)

        # =========================
        # CLASSICAL PATH
        # =========================
        classical_logits = self.classical_head(x)

        # =========================
        # QUANTUM PATH
        # =========================

        # Encode
        h0 = torch.tanh(self.encoder(x)) * math.pi

        # Add small noise (stabilization)
        h0 = h0 + 0.01 * torch.randn_like(h0)

        # Evolution step (residual)
        h1 = h0 + self.mix(torch.cat([h0, h0], dim=1))

        # Stack evolution inputs
        inputs_list = torch.stack([h0, h1], dim=1)

        # Apply quantum circuit (step-wise evolution)
        q_out = torch.stack([
            torch.stack([
                torch.stack(qnode(inputs_list[i][step], self.q_weights[step]))
                for step in range(n_steps)
            ]).mean(dim=0)
            for i in range(len(inputs_list))
        ])

        # Decode quantum output
        quantum_logits = self.decoder(q_out)

        # =========================
        # HYBRID COMBINATION
        # =========================
        alpha = torch.sigmoid(self.alpha)

        logits = alpha * quantum_logits + (1 - alpha) * classical_logits

        return logits

# =========================
# TRAIN FUNCTION (LIVE METRICS)
# =========================
def train(model, loader, optimizer, loss_fn, epochs):

    model.train()

    for epoch in range(epochs):

        total_loss = 0
        correct = 0
        total = 0

        pbar = tqdm(loader, desc=f"Epoch {epoch+1}/{epochs}", leave=False)

        for xb, yb in pbar:

            xb, yb = xb.to(device), yb.to(device)

            optimizer.zero_grad()

            logits = model(xb)
            loss = loss_fn(logits, yb)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            total_loss += loss.item()

            preds = logits.argmax(dim=1)
            correct += (preds == yb).sum().item()
            total += yb.size(0)

            acc = correct / total

            alpha_val = "-"
            if hasattr(model, "alpha"):
                alpha_val = f"{torch.sigmoid(model.alpha).item():.2f}"

            pbar.set_postfix({
                "loss": f"{loss.item():.3f}",
                "avg_loss": f"{total_loss/(total//yb.size(0)+1):.3f}",
                "acc": f"{acc*100:.2f}%",
                "alpha": alpha_val
            })

        print(f"\nEpoch {epoch+1} Summary:")
        print(f"Loss: {total_loss/len(loader):.4f}")
        print(f"Accuracy: {acc*100:.2f}%")

        if hasattr(model, "alpha"):
            print(f"Alpha: {torch.sigmoid(model.alpha).item():.4f}")

# EVALUATE
# =========================
def evaluate(model, loader, noise=False):

    model.eval()
    correct, total = 0, 0

    with torch.no_grad():
        for xb, yb in loader:

            xb, yb = xb.to(device), yb.to(device)

            if noise:
                xb = xb + 0.05 * torch.randn_like(xb)

            preds = model(xb).argmax(dim=1)

            correct += (preds == yb).sum().item()
            total += yb.size(0)

    return correct / total

# =========================
# RUN EXPERIMENT
# =========================
models = {
    "Classical": ClassicalModel(),
    "v8.1": ModelV81()
}

results = {}

for name, model in models.items():

    print(f"\n🚀 Training {name}\n")

    model = model.to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.CrossEntropyLoss()

    train(model, train_loader, optimizer, loss_fn, epochs)

    acc_train = evaluate(model, train_loader)
    acc_test = evaluate(model, test_loader)
    acc_noise = evaluate(model, test_loader, noise=True)

    results[name] = (acc_train, acc_test, acc_noise)

# =========================
# FINAL RESULTS
# =========================
print("\n📊 FINAL COMPARISON\n")

for name, (tr, te, no) in results.items():
    print(f"{name:10}")
    print(f"  Train Accuracy : {tr*100:.2f}%")
    print(f"  Test Accuracy  : {te*100:.2f}%")
    print(f"  Noise Accuracy : {no*100:.2f}%\n")

